# Phase 111: replication and decoded recovery on IBM hardware

Runs the frozen Phase 111 design (`DESIGN.md`) on ibm_fez. Nothing here compiles, chooses or checks circuits by
itself: all run logic is in `p111_runner.py`, which is frozen and hashed with everything else; this notebook checks
that file's hash and then only calls it.

**Jobs, in order, each submitted once:**
1. **Part A2**, replication of Test B: the archived compiled circuits (`testB_transpiled.qpy`), 20 × 4,000 shots.
2. **Part B pilot**: the six circuits of D1 and D2 plus their calibrations, 1,000 shots each. No abort on the values;
   only a technical failure stops the run.
3. **Part B**: all decoding, baseline and calibration circuits, at the shot count fixed by the power check.

**Before running:**
- Put the frozen files (listed in `FREEZE_MANIFEST.txt`) in `MyDrive/phase111`.
- Verify the OpenTimestamps proof of `FREEZE_MANIFEST.txt` (for example at opentimestamps.org) and put the result in
  `TIMESTAMP_CHECK.json`: `{"manifest_sha256": ..., "attested_utc": ..., "bitcoin_block": ..., "verified_with": ...,
  "verified_utc": ...}`.
- Have the manifest's SHA-256 from the timestamped commit at hand, and set `INSTANCE` in the connection cell.

**Safeguards** (in `p111_runner.py`): every frozen file must match the timestamped manifest; the experiment ID derived
from it is in every job tag and job record; just before each submission all hashes, today's target, the shot count and
the order of steps are checked again and you confirm the QPU budget; retrieval checks the job's tag, backend, shots and
submitted circuits. After a disconnect, rerun the cells up to the connection, then the cell of the step you were on;
nothing is resubmitted. After the run, save a copy of this executed notebook with the results.

In [ ]:
# The versions that built the frozen circuits: QPY files need a Qiskit at least as new as the writer.
!pip -q install "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0"

In [ ]:
import os, sys, hashlib, getpass
OUTDIR = '/content/drive/MyDrive/phase111'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:                       # not in Colab: use a local folder of the same name
    print('Drive not mounted:', e)
    OUTDIR = 'phase111'
print('working folder:', OUTDIR)

## Check the frozen files
Checks `p111_runner.py` and the modules it imports against the manifest before importing them; the runner then checks
everything else.

In [ ]:
STAMPED = input('SHA-256 of FREEZE_MANIFEST.txt in the timestamped commit: ').strip().lower()
_sha = lambda n: hashlib.sha256(open(os.path.join(OUTDIR, n), 'rb').read()).hexdigest()
assert _sha('FREEZE_MANIFEST.txt') == STAMPED, 'this FREEZE_MANIFEST.txt is not the timestamped one'
_listed = {}
for _line in open(os.path.join(OUTDIR, 'FREEZE_MANIFEST.txt')):
    if _line.strip() and not _line.startswith('#'):
        _h, _n = _line.split(None, 1)
        _listed[_n.strip()] = _h
for _f in ('p111_runner.py', 'p111_verify.py', 'p111_analyse.py'):
    assert _listed.get(_f) == _sha(_f), f'{_f} does not match the manifest'
sys.path.insert(0, OUTDIR)
import p111_runner as RUN
EXP = RUN.Experiment(OUTDIR, STAMPED)

## Connect to IBM Quantum
Uses the `IBM_QUANTUM_TOKEN` environment variable if set, otherwise asks for your API key (not stored). Set
`INSTANCE` to your own instance CRN.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
CHANNEL = "ibm_cloud"
INSTANCE = "<your IBM Quantum instance CRN>"
token = os.environ.get('IBM_QUANTUM_TOKEN') or getpass.getpass("IBM Quantum API key: ")
EXP.connect(QiskitRuntimeService(channel=CHANNEL, token=token, instance=INSTANCE), "ibm_fez", SamplerV2)

## Target check (no QPU time)
Reports whether today's backend supports every frozen circuit (checked again before each submission). If the archived
Test B circuits fail, Part A2 needs the A2b procedure; if the Part B circuits fail, they are recompiled offline by the
frozen procedure, re-verified and re-stamped. This notebook does not recompile.

In [ ]:
EXP.target_report()

## Part A2: replication of Test B (archived circuits)

In [ ]:
EXP.submit_once('A2')

In [ ]:
A2_OUT = EXP.analyse('A2')        # retrieve and analyse (safe to rerun)

## Part B pilot (fixed; no abort on the values)

In [ ]:
EXP.submit_once('pilot')

In [ ]:
P_OUT = EXP.analyse('pilot')      # only a technical failure (an error raised here) stops the run

## Part B: decoded recovery

In [ ]:
EXP.submit_once('B')

In [ ]:
B_OUT = EXP.analyse('B')          # retrieve and analyse (safe to rerun)

## Job record
Lists every job on this instance since a day before now (any status), for `RUNLOG.md`.

In [ ]:
EXP.job_record()